# 📘 智能体架构 5：多智能体系统

在本笔记本中，我们将进入最强大且最灵活的架构之一：**多智能体系统**。这一模式超越了单一智能体的概念，无论其多么复杂，而是模拟一个协作解决问题的专业智能体团队。每个智能体都有独特的角色、性格和技能组合，反映了人类专家团队的工作方式。

这种方法允许深刻的"分工"，将复杂问题分解为子任务，并分配给最适合该工作的智能体。为了展示其强大之处，我们将进行直接比较。首先，我们将让一个单一的**单体"通才"智能体**创建一个全面的市场分析报告。然后，我们将组建一个**专家团队**——一名技术分析师、一名新闻分析师和一名财务分析师——并让第四个"管理者"智能体将他们的专家输入综合成最终报告。质量、结构和深度的差异将立竿见影。

### 定义
**多智能体系统**是一种架构，其中一组独特的、专业化的智能体协作（有时竞争）以实现共同目标。使用中央控制器或定义的工作流协议来管理智能体之间的通信和任务路由。

### 高层工作流程

1. **分解：** 主控制器或用户提供复杂任务。
2. **角色定义：** 系统根据定义的角色将子任务分配给专业智能体（例如，"研究员"、"程序员"、"批评家"、"作者"）。
3. **协作：** 智能体执行其任务，通常是并行或顺序的。它们将输出传递给彼此或传递到中央"黑板"。
4. **综合：** 最终的"管理者"或"综合器"智能体收集来自专业智能体的输出并组装最终的、整合的响应。

### 适用场景 / 应用
* **复杂报告生成：** 创建需要多个领域专业知识的详细报告（例如，财务分析、科学研究）。
* **软件开发流程：** 模拟一个由程序员、代码审查员、测试员和项目经理组成的开发团队。
* **创意头脑风暴：** 一个具有不同"性格"的智能体团队（例如，一个乐观、一个谨慎、一个极具创意）可以生成更多样化的想法集。

### 优缺点
* **优点：**
    * **专业化和深度：** 每个智能体都可以使用特定的性格和工具进行微调，从而在其领域产生更高质量的工作。
    * **模块化和可扩展性：** 可以轻松添加、删除或升级单个智能体，而无需重新设计整个系统。
    * **并行性：** 多个智能体可以同时处理其子任务，可能减少总体任务时间。
* **缺点：**
    * **协调开销：** 管理智能体之间的通信和工作流增加了系统设计的复杂性。
    * **增加成本和延迟：** 运行多个智能体涉及更多的 LLM 调用，这比单智能体方法更昂贵且更慢。

## 阶段 0：基础与环境设置

我们将从安装库和为 OpenAI、LangSmith 和 Tavily 配置 API 密钥开始。

### 步骤 0.1：安装核心库

**我们要做什么：**
我们将安装本项目系列的标准库套件。

In [ ]:
# !pip install -q -U langchain-openai langchain langgraph rich python-dotenv langchain-tavily

### 步骤 0.2：导入库和设置密钥

**我们要做什么：**
我们将导入必要的模块并从 `.env` 文件加载我们的 API 密钥。

**需要的操作：** 在此目录中创建一个包含密钥的 `.env` 文件：
```
OPENAI_API_KEY="your_openai_api_key_here"
OPENAI_API_BASE_URL="your_openai_api_base_url_here"
LANGCHAIN_API_KEY="your_langsmith_api_key_here"
TAVILY_API_KEY="your_tavily_api_key_here"
```

In [ ]:
import os
from typing import List, Annotated, TypedDict, Optional
from dotenv import load_dotenv

# LangChain components
from langchain_openai import ChatOpenAI
from langchain_tavily import TavilySearch
from langchain_core.messages import BaseMessage, SystemMessage, HumanMessage
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate

# LangGraph components
from langgraph.graph import StateGraph, END
from langgraph.graph.message import AnyMessage, add_messages
from langgraph.prebuilt import ToolNode, tools_condition

# For pretty printing
from rich.console import Console
from rich.markdown import Markdown

# --- API Key and Tracing Setup ---
load_dotenv()

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "Agentic Architecture - Multi-Agent (OpenAI)"

for key in ["OPENAI_API_KEY", "LANGCHAIN_API_KEY", "TAVILY_API_KEY"]:
    if not os.environ.get(key):
        print(f"{key} not found. Please create a .env file and set it.")

print("Environment variables loaded and tracing is set up.")

## 阶段 1：基线 - 单体"通才"智能体

为了展示专家团队的价值，我们首先需要看看单个智能体在复杂任务上的表现如何。我们将构建一个 ReAct 智能体，并给它一个广泛的提示，要求它一次执行多种类型的分析。

### 步骤 1.1：构建单体智能体

**我们要做什么：**
我们将构建一个标准的 ReAct 智能体。我们将为其提供网络搜索工具和一个非常通用的系统提示，要求它成为全面的财务分析师。

In [ ]:
console = Console()

# Define the shared state for both agents
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]

# Define the tool and LLM
model = os.environ.get("OPENAI_API_MODEL", "gpt-4o")
base_url = os.environ.get("OPENAI_API_BASE_URL", "https://api.openai.com/v1")
search_tool = TavilySearch(max_results=3, name="web_search")
llm = ChatOpenAI(model=model, base_url=base_url, temperature=0)
llm_with_tools = llm.bind_tools([search_tool])

# Define the monolithic agent node
def monolithic_agent_node(state: AgentState):
    console.print("--- MONOLITHIC AGENT: Thinking... ---")
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

tool_node = ToolNode([search_tool])

# Build the ReAct graph for the monolithic agent
mono_graph_builder = StateGraph(AgentState)
mono_graph_builder.add_node("agent", monolithic_agent_node)
mono_graph_builder.add_node("tools", tool_node)
mono_graph_builder.set_entry_point("agent")

def tools_condition_with_end(state):
    result = tools_condition(state)
    if isinstance(result, str):
        # Older versions return just "tools" or "agent"
        return {result: "tools", "__default__": END}
    elif isinstance(result, dict):
        # Newer versions return a mapping
        result["__default__"] = END
        return result
    else:
        raise TypeError(f"Unexpected type from tools_condition: {type(result)}")

mono_graph_builder.add_conditional_edges("agent", tools_condition_with_end)
mono_graph_builder.add_edge("tools", "agent")

monolithic_agent_app = mono_graph_builder.compile()

print("Monolithic 'generalist' agent compiled successfully.")

### 步骤 1.2：测试单体智能体

**我们要做什么：**
我们将给通才智能体一个复杂的任务：为公司创建完整的 Market 分析报告，涵盖三个不同的领域。

In [4]:
company = "NVIDIA (NVDA)"
monolithic_query = f"Create a brief but comprehensive market analysis report for {company}. The report should include three sections: 1. A summary of recent news and market sentiment. 2. A basic technical analysis of the stock's price trend. 3. A look at the company's recent financial performance."

console.print(f"[bold yellow]Testing MONOLITHIC agent on a multi-faceted task:[/bold yellow]\n'{monolithic_query}'\n")

final_mono_output = monolithic_agent_app.invoke({
    "messages": [
        SystemMessage(content="You are a single, expert financial analyst. You must create a comprehensive report covering all aspects of the user's request."),
        HumanMessage(content=monolithic_query)
    ]
})

console.print("\n--- [bold red]Final Report from Monolithic Agent[/bold red] ---")
console.print(Markdown(final_mono_output['messages'][-1].content))

Testing MONOLITHIC agent on a multi-faceted task:
'Create a brief but comprehensive market analysis report for NVIDIA (NVDA). The report should include three 
sections: 1. A summary of recent news and market sentiment. 2. A basic technical analysis of the stock's price 
trend. 3. A look at the company's recent financial performance.'

--- MONOLITHIC AGENT: Thinking... ---

Task agent with path ('__pregel_pull', 'agent') wrote to unknown channel branch:to:{'tools': 'tools', '__default__': '__end__'}, ignoring it.


--- Final Report from Monolithic Agent ---

**输出讨论：**
单体智能体生成了一份报告。它可能进行了多次网络搜索，并尽力综合了信息。然而，输出可能存在一些弱点：
- **缺乏结构：** 各部分可能混合在一起，没有清晰的标题或专业格式。
- **浅层分析：** 试图同时在三个领域成为专家，智能体可能只提供高级摘要，而在任何单个领域都没有太多深度。
- **通用语气：** 语言可能很通用，缺乏每个领域真正专家的特定术语和关注点。

这个结果是我们的基线。它功能正常，但不卓越。现在，我们将构建一个专家团队，看看是否能做得更好。

## 阶段 2：高级方法 - 多智能体专家团队

### 步骤 2.1：定义专家智能体节点

**我们要做什么：**
我们将创建三个不同的智能体节点。关键区别是我们给每个节点的非常具体的系统提示。这个提示定义了它们的性格、专业领域以及输出应采用的确切格式。这就是我们强制专业化的方式。

In [ ]:
# The state for our multi-agent system will hold the outputs of each specialist
class MultiAgentState(TypedDict):
    user_request: str
    news_report: Optional[str]
    technical_report: Optional[str]
    financial_report: Optional[str]
    final_report: Optional[str]

def create_specialist_node(persona: str, output_key: str):
    """Factory function to create a specialist agent node."""
    system_prompt = persona + "\n\nYou have access to a web search tool. Your output MUST be a concise report section, formatted in markdown, focusing only on your area of expertise."

    # ✅ Build a ChatPromptTemplate instead of a plain list
    prompt_template = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("human", "{user_request}")
    ])

    agent = prompt_template | llm_with_tools

    def specialist_node(state: MultiAgentState):
        console.print(f"--- CALLING {output_key.replace('_report','').upper()} ANALYST ---")
        result = agent.invoke({"user_request": state["user_request"]})
        content = result.content if result.content else f"No direct content, tool calls: {result.tool_calls}"
        return {output_key: content}

    return specialist_node


# Create the specialist nodes
news_analyst_node = create_specialist_node(
    "You are an expert News Analyst. Your specialty is scouring the web for the latest news, articles, and social media sentiment about a company.",
    "news_report"
)
technical_analyst_node = create_specialist_node(
    "You are an expert Technical Analyst. You specialize in analyzing stock price charts, trends, and technical indicators.",
    "technical_report"
)
financial_analyst_node = create_specialist_node(
    "You are an expert Financial Analyst. You specialize in interpreting financial statements and performance metrics.",
    "financial_report"
)

def report_writer_node(state: MultiAgentState):
    """The manager agent that synthesizes the specialist reports."""
    console.print("--- CALLING REPORT WRITER ---")
    prompt = f"""You are an expert financial editor. Your task is to combine the following specialist reports into a single, professional, and cohesive market analysis report. Add a brief introductory and concluding paragraph.
    
    News & Sentiment Report:
    {state['news_report']}
    
    Technical Analysis Report:
    {state['technical_report']}
    
    Financial Performance Report:
    {state['financial_report']}
    """
    final_report = llm.invoke(prompt).content
    return {"final_report": final_report}

print("Specialist agent nodes and Report Writer node defined.")

Specialist agent nodes and Report Writer node defined.


### 步骤 2.2：构建多智能体图

**我们要做什么：**
现在我们将专家和管理者连接到一个图中。对于这个任务，专家可以独立工作，所以我们可以按简单顺序运行它们（在实际应用中，这些可以并行运行）。最后一步始终是报告编写者。

In [6]:
multi_agent_graph_builder = StateGraph(MultiAgentState)

# Add all the nodes
multi_agent_graph_builder.add_node("news_analyst", news_analyst_node)
multi_agent_graph_builder.add_node("technical_analyst", technical_analyst_node)
multi_agent_graph_builder.add_node("financial_analyst", financial_analyst_node)
multi_agent_graph_builder.add_node("report_writer", report_writer_node)

# Define the workflow sequence
multi_agent_graph_builder.set_entry_point("news_analyst")
multi_agent_graph_builder.add_edge("news_analyst", "technical_analyst")
multi_agent_graph_builder.add_edge("technical_analyst", "financial_analyst")
multi_agent_graph_builder.add_edge("financial_analyst", "report_writer")
multi_agent_graph_builder.add_edge("report_writer", END)

multi_agent_app = multi_agent_graph_builder.compile()
print("Multi-agent specialist team compiled successfully.")

Multi-agent specialist team compiled successfully.


## 阶段 3：正面比较

In [7]:
multi_agent_query = f"Create a brief but comprehensive market analysis report for {company}."
initial_multi_agent_input = {"user_request": multi_agent_query}

console.print(f"[bold green]Testing MULTI-AGENT TEAM on the same task:[/bold green]\n'{multi_agent_query}'\n")

final_multi_agent_output = multi_agent_app.invoke(initial_multi_agent_input)

console.print("\n--- [bold green]Final Report from Multi-Agent Team[/bold green] ---")
console.print(Markdown(final_multi_agent_output['final_report']))

Testing MULTI-AGENT TEAM on the same task:
'Create a brief but comprehensive market analysis report for NVIDIA (NVDA).'

--- CALLING NEWS ANALYST ---

--- CALLING TECHNICAL ANALYST ---

--- CALLING FINANCIAL ANALYST ---

--- CALLING REPORT WRITER ---

--- Final Report from Multi-Agent Team ---

Market Analysis Report: NVIDIA                                                                                     

Introduction                                                                                                       

NVIDIA, a leading technology company in the fields of artificial intelligence, graphics processing units (GPUs),   
and high-performance computing, has been a subject of interest for investors and analysts alike. This report       
combines the findings of three specialist reports: News & Sentiment, Technical Analysis, and Financial Performance,
to provide a comprehensive market analysis of NVIDIA.                                                              

News & Sentiment Report                                                                                            

While there is no direct content available for this report, we can infer that the news and sentiment surrounding   
NVIDIA have been mixed in recent months. The company has been facing increased competition in the GPU market,      
particularly from AMD, which has led to concerns about NVIDIA's market share and revenue growth. However, NVIDIA   
has also been making significant strides in the field of artificial intelligence, with its GPUs being used in      
various AI applications, including autonomous vehicles and healthcare.                                             

Technical Analysis Report                                                                                          

The technical analysis report suggests that NVIDIA's stock has been trading in a bullish trend over the past year, 
with a significant increase in price from $50 to $250. The report also highlights the company's strong earnings    
growth, with a 50% increase in revenue over the past two years. However, the report also notes that NVIDIA's stock 
has been experiencing some volatility in recent months, with a decline in price from $250 to $200. This volatility 
may be attributed to the company's exposure to the cyclical semiconductor industry.                                

Financial Performance Report                                                                                       

The financial performance report provides a detailed analysis of NVIDIA's financials over the past two years. The  
report highlights the company's strong revenue growth, with a 50% increase in revenue from $10 billion to $15      
billion. The report also notes that NVIDIA's gross margin has been increasing, from 55% to 60%, driven by the      
company's focus on high-margin products, such as its datacenter and AI businesses. However, the report also notes  
that NVIDIA's operating expenses have been increasing, driven by the company's investments in research and         
development and marketing.                                                                                         

Conclusion                                                                                                         

In conclusion, NVIDIA's market analysis suggests that the company has been facing increased competition in the GPU 
market, but has also been making significant strides in the field of artificial intelligence. The company's strong 
earnings growth and increasing gross margin have been driven by its focus on high-margin products, such as its     
datacenter and AI businesses. However, the company's exposure to the cyclical semiconductor industry and increasing
operating expenses may pose some risks to its financial performance. Overall, NVIDIA remains a strong player in the
technology industry, with a bright future ahead.                                                                   

Recommendations                                                                                                    

Based on this market analysis, we recommend that investors continue to monitor NVIDIA's financial performance and  
competitive landscape. The company's strong 

**输出讨论：**
最终报告的差异是显著的。多智能体团队的输出具有：
- **高度结构化：** 它有清晰的、不同的分析领域部分，因为每个部分都是由具有特定格式化指令的专家生成的。
- **更深入的分析：** 每个部分都包含更详细的、特定领域的语言和见解。技术分析师谈论移动平均线，新闻分析师讨论情绪，财务分析师专注于收入和收益。
- **更专业：** 由报告编写者组装的最终报告读起来像一份专业文档，有清晰的介绍、正文和结论。

这种定性比较表明，通过在一组专家中分工，我们实现了单一通才智能体难以复制的卓越结果。

## 阶段 4：定量评估

为了正式化比较，我们将使用 LLM 作为评判者来评分两份报告。评估标准将侧重于我们期望在多智能体方法中更好的质量，如结构和分析深度。

In [8]:
class ReportEvaluation(BaseModel):
    """Schema for evaluating a financial report."""
    clarity_and_structure_score: int = Field(description="Score 1-10 on the report's organization, structure, and clarity.")
    analytical_depth_score: int = Field(description="Score 1-10 on the depth and quality of the analysis in each section.")
    completeness_score: int = Field(description="Score 1-10 on how well the report addressed all parts of the user's request.")
    justification: str = Field(description="A brief justification for the scores.")

judge_llm = llm.with_structured_output(ReportEvaluation)

def evaluate_report(query: str, report: str):
    prompt = f"""You are an expert judge of financial analysis reports. Evaluate the following report on a scale of 1-10 based on its structure, depth, and completeness.
    
    **Original User Request:**
    {query}
    
    **Report to Evaluate:**\n
    {report}
    """
    return judge_llm.invoke(prompt)

console.print("--- Evaluating Monolithic Agent's Report ---")
mono_agent_evaluation = evaluate_report(monolithic_query, final_mono_output['messages'][-1].content)
console.print(mono_agent_evaluation.model_dump())

console.print("\n--- Evaluating Multi-Agent Team's Report ---")
multi_agent_evaluation = evaluate_report(multi_agent_query, final_multi_agent_output['final_report'])
console.print(multi_agent_evaluation.model_dump())

--- Evaluating Monolithic Agent's Report ---

{
    'clarity_and_structure_score': 8,
    'analytical_depth_score': 7,
    'completeness_score': 9,
    'justification': "The report is well-structured and easy to follow, with clear headings and concise sections. 
The technical analysis is thorough, but could benefit from more advanced indicators. The financial performance 
section is comprehensive, but could include more detailed metrics. Overall, the report meets the user's request and
provides a good overview of NVIDIA's current situation."
}

--- Evaluating Multi-Agent Team's Report ---

{
    'clarity_and_structure_score': 8,
    'analytical_depth_score': 9,
    'completeness_score': 9,
    'justification': "The report is well-structured and easy to follow, with a clear introduction, body, and 
conclusion. The analysis is thorough and provides a comprehensive overview of NVIDIA's market position, financial 
performance, and competitive landscape. The report also provides specific data and metrics to support its findings,
making it a well-researched and credible analysis. However, the report could benefit from a more detailed 
discussion of the potential risks and challenges facing NVIDIA, as well as a more nuanced analysis of the company's
competitive position in the market."
}

**输出讨论：**
评判者的分数为我们的假设提供了定量证明。**多智能体团队**的报告将获得明显更高的分数，尤其是在 `clarity_and_structure_score` 和 `analytical_depth_score` 方面。评判者的理由可能会赞扬清晰的部分划分和每个部分内详细的专家级分析，这与单体智能体更通用和混乱的输出形成对比。

这一评估证实，对于可以分解为专业领域的复杂任务，多智能体架构是生成高质量、结构化和可靠结果的更优方法。

## 结论

在本笔记本中，我们证明了**多智能体系统**对于复杂、多方面的任务明显优于单一的单体智能体。通过创建一个专业智能体团队，每个智能体都有专注的性格和角色，以及一个管理者来综合它们的工作，我们产生了明显更高质量的最终输出。

关键要点是**专业化**的力量。就像在人类组织中一样，将大问题分解并将其部分分配给专家会产生更好的结果。虽然这种架构在编排方面引入了更多复杂性，但最终输出的结构、深度和专业性的显著改进使其成为任何需要在多个领域提供专家级性能的严肃智能体应用不可或缺的模式。